# AlexNet model Implementation using PyTorch 

> Implemetation of neural network architecture, AlexNet in PyTorch. Authors of the paper are, Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton.

- toc:true
- branch: master
- badges: true
- author: Pratik Kumar
- categories: [PyTorch, CNN, Image Classification, Computer Vision]

## 1. **Introduction** ☕️
****
In this notebook, we will be implementing AlexNet from scratch using PyTorch. This is first from the series of blogs on Convolutional Neural Networks(CNN). Here we will be implementing common CNN architectures (like ResNet,VGG). We will be using PyTorch to code from scratch. This series aims to develop a deep understanding of the Convolutional neural networks. To know more details follow the mentioned official links in 'References' section.

In [29]:
#IMPORTING NECESSARY LIBRARIES
import os
import time
    
import torch
import torch.nn as nn
from torchsummary import summary

## 2. **Deep Convolutional Neural Network** 🤖
****
After the first CNN, [LeNet-5](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf), introduced by Yann LeCun, the next major deep neural network is considered to be AlexNet. It is famously known for its victory in 2012 ImageNet LSVRC-2012 competition. AlexNet is one of those early architectures that was trained on GPUs. The network is similar to LeNet but has more filters, convolutions, max-pooling, dropout, data augmentations, ReLU activations and SGD momentum. These operations have a significant effect over the performance of a CNN.
 
The authors of AlexNet used the ImageNet dataset to train the network. This dataset consisted of over 15 million labelled high-resolution images, belonging roughly 22,000 categories. The network showed, for the first time, that the features obtained by learning can go beyond manually-designed features. This broke the previous paradigm in computer vision. Before this, traditional machine learning models like SVM outperformed CNN.

It was found that, AlexNet covered two of the limitations for general image classification in real world. Following are those problems faced and solution implied from AlexNet paper: 
 
1. Lack of computational resources - Training on parallel processing processors such as GPUs.  
2. Lack of robust dataset - Using variety of images and large datasets (such as ImageNet dataset).

Researchers used to train the models on smaller datasets such as MNIST Handwritten digits. This was not enough to cover the complete image classification task. Training on GPUs turned out to be very fast due to their parallel processing tehnique. These were important ingredients that were missing before AlexNet was introduced. Today almost every neural network architecture is trained on GPUs with large datasets. AlexNet was the point in deep learning history that made a significant impact in growing of interest in CNNs.


### 2.1. **About Model**
---
![](https://github.com/pr2tik1/blog/blob/master/_notebooks/images/AlexNet-1.png?raw=1 "Credit: https://neurohive.io/en/popular-networks/alexnet-imagenet-classification-with-deep-convolutional-neural-networks/")

The model has 5 Convolution layers and 3 fully connected layers, i.e. total 8 layers. Following is a general overview of layers.

- Types of layers and about them: 
    - *Convolution* (5 layers) : These layers perform convolution operations along with striding and padding.
    - *Max Pooling* (3 times) : Max-Pooling operation is performed. 
    - *Normalisation* (2 times) : Normalisation to compensate the unbounded nature of activation functions such as ReLU.
    - *Fully Connected* (3 layers) : These are at the end which finally give the outputs in n-classes (in this case 1000).

- The trainable parameters are in 8 layers (5 Conv. + 3 F.C.) of the model. Hence we do not count Max-Pooling and Normalisation as layers.

- A table of layers and their characterstics, with reference to the proposed architecture in paper,

![](https://github.com/pr2tik1/blog/blob/master/_notebooks/images/AlexNet.png?raw=1)

### 2.3 Coding the AlexNet Model
---
The alexnet class is divided into two subclasses: **features** and **classifier**. The *features subclass* has 5 convolution layers with pooling and normalisation operations. Here ReLU is used as activation function. The *classifier subclass* consists the three fully connected layers with the ReLU activation and dropout operation. Final output is 1000 as it denotes the 1000 classes.

In [38]:
class alexnet(nn.Module):
    def __init__(self, num_classes: int=1000)->None:
        super(alexnet, self).__init__()
        self.features = nn.Sequential(
            #Layer 1
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4),#Convolution
            nn.ReLU(),#Activation Function
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),#Normalisation
            nn.MaxPool2d(kernel_size=3, stride=2),#MaxPooling
            #Layer 2
            nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            #Layer 3
            nn.Conv2d(in_channels=256 , out_channels=384, kernel_size=3, padding=1),
            nn.ReLU(),
            #Layer 4
            nn.Conv2d(in_channels=384 , out_channels=384, kernel_size=3, padding=1),
            nn.ReLU(),
            #Layer 5
            nn.Conv2d(in_channels=384 , out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2)   
        )

        
        self.classifier = nn.Sequential(
            #Layer 6
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(in_features=(256 * 6 * 6), out_features=4096),
            nn.ReLU(),
            #Layer 7
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(in_features=4096, out_features=4096),
            nn.ReLU(),
            #Layer 8
            nn.Linear(in_features=4096, out_features=num_classes)
        )

    def forward(self, x: torch.Tensor)-> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x


In [39]:
model = alexnet()
model

alexnet(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): ReLU()
    (2): LocalResponseNorm(5, alpha=0.0001, beta=0.75, k=2)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (5): ReLU()
    (6): LocalResponseNorm(5, alpha=0.0001, beta=0.75, k=2)
    (7): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=True)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2)

#### 2.3.1 Operations 
--- 

Understanding few operations as observed above,

1. **Convolution**: A convolution is application of a filter to an input that results in an activation. Repeatedely use of the same filter to an input results in a map of activations called a **feature map**.

2. **Activation Function**: Function that limits or fires the weighted sum of input into an output based upon its mathematical behaviour within a neural network. 

3. **Normalization**: To change the values of numeric columns in a dataset to use a common scale when the features in the data have different ranges.

4. **Max Pooling**: Pooling layers are used to reduce the dimensions of the feature maps. Here we used Max-Pooling among the available pooling techniques.

5. **Dropout**:  The term “dropout” refers to dropping out units (hidden and visible) in a neural network. Used to prevent overfitting.


### 2.4. Model Summary
---
Below is summary of our model that gives the total trainable parameters and dimensions for each layer. This can be compared with the originally proposed AlexNet.


In [40]:
summary(model, (3, 227, 227))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 96, 55, 55]          34,944
              ReLU-2           [-1, 96, 55, 55]               0
 LocalResponseNorm-3           [-1, 96, 55, 55]               0
         MaxPool2d-4           [-1, 96, 27, 27]               0
            Conv2d-5          [-1, 256, 27, 27]         614,656
              ReLU-6          [-1, 256, 27, 27]               0
 LocalResponseNorm-7          [-1, 256, 27, 27]               0
         MaxPool2d-8          [-1, 256, 13, 13]               0
            Conv2d-9          [-1, 384, 13, 13]         885,120
             ReLU-10          [-1, 384, 13, 13]               0
           Conv2d-11          [-1, 384, 13, 13]       1,327,488
             ReLU-12          [-1, 384, 13, 13]               0
           Conv2d-13          [-1, 256, 13, 13]         884,992
             ReLU-14          [-1, 256,

Notice the size of model, total trainable parameter and parameter size. This can be compared with LeNet Classifier in my previous post [here](https://pr2tik1.github.io/blog/pytorch/cnn/pca/t-sne/image%20classification/computer%20vision/2020/09/08/Sketch-Recognition.html#2.1.3.-Model-Summary).

## 3. **Conclusion** 😇
---

We have learnt about AlexNet and its layers by implementation from scratch. Next in series we will cover other common CNN architecture. Image Classification task is not easy. The total duration needed to train AlexNet on 2 GPUs of NVIDIA is said to be around 3 days! Moreover, GPUs outperform CPUs and brought a whole new world of better computational power in deep learning.



## ***References*** 📜
---

- Original Paper: 
  -  https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks.pdf 

- Machine Learning Mastery: 
  -  https://machinelearningmastery.com/convolutional-layers-for-deep-learning-neural-networks/
  -  https://machinelearningmastery.com/choose-an-activation-function-for-deep-learning/

- [AlexNet Pytorch Tochvision model](https://github.com/pytorch/vision/blob/master/torchvision/models/alexnet.py)

- [PyTorch Installation Guide](https://pytorch.org/get-started/locally/)

- [PyTorch Official Tutorials](https://pytorch.org/tutorials/)

- [Yannic's explanation](https://www.youtube.com/watch?v=Nq3auVtvd9Q)

# ***Thank you***! 🙏
